In [0]:
%run ../config/00_config

In [0]:
%run ../utils/00_utils

In [0]:
adls_options = build_adls_options(
    storage_account_name=ADLS_STORAGE_ACCOUNT_NAME,
    client_id=ADLS_CLIENT_ID,
    tenant_id=ADLS_TENANT_ID,
    client_secret=ADLS_CLIENT_SECRET,
)

print(f"Arquivo origem: {SOURCE_FILE}")
print(f"Tabela destino: {TARGET_FULL_TABLE}")

In [0]:
df_clientes = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .options(**adls_options)
    .load(SOURCE_PATH)
)

print("Arquivo CSV lido com sucesso.")

In [0]:
expected_columns = [
    "id_cliente",
    "uuid_cliente",
    "nome",
    "sobrenome",
    "email",
    "senha_hash",
    "dt_cadastro",
    "dt_ultima_atualizacao",
]

unexpected_columns = validate_required_columns(df_clientes, expected_columns)

total_linhas = df_clientes.count()

assert total_linhas > 0, "A tabela ecommerce_clientes foi lida sem registros."

print("Schema validado com sucesso.")
print(f"Total de linhas lidas: {total_linhas}")
print(f"Colunas adicionais encontradas: {unexpected_columns}")

In [0]:
write_sql_table(
    df=df_clientes,
    sql_host=SQL_HOST,
    sql_database=SQL_DATABASE,
    sql_username=SQL_USERNAME,
    sql_password=SQL_PASSWORD,
    table_name=TARGET_FULL_TABLE,
    mode="overwrite",
)

print(f"Tabela gravada com sucesso no SQL Server: {TARGET_FULL_TABLE}")

In [0]:
df_validacao_sql = read_sql_table(
    spark=spark,
    sql_host=SQL_HOST,
    sql_database=SQL_DATABASE,
    sql_username=SQL_USERNAME,
    sql_password=SQL_PASSWORD,
    table_name=TARGET_FULL_TABLE,
)

total_gravado_sql = df_validacao_sql.count()

print(f"Total de registros na origem: {total_linhas}")
print(f"Total de registros gravados em {TARGET_FULL_TABLE}: {total_gravado_sql}")

assert total_gravado_sql == total_linhas, (
    "Divergência entre quantidade de registros na origem e no SQL Server."
)